In [1]:
import torchaudio as ta
from chatterbox.tts import ChatterboxTTS

model = ChatterboxTTS.from_pretrained(device="cuda")

text = "Ezreal and Jinx teamed up with Ahri [LAUGH], Yasuo, and Teemo to take down the enemy's Nexus in an epic late-game pentakill."
# wav = model.generate(text)
# ta.save("test-1.wav", wav, model.sr)

# If you want to synthesize with a different voice, specify the audio prompt
AUDIO_PROMPT_PATH="/home/user/voice/data/styletts/audio/train_data_audio/angry_Anger-1.wav"
wav = model.generate(text, audio_prompt_path=AUDIO_PROMPT_PATH)
ta.save("/home/user/voice/chatterbox/infer-1.wav", wav, model.sr)

/home/user/anaconda3/envs/chatterbox/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/user/anaconda3/envs/chatterbox/lib/python3.11/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


loaded PerthNet (Implicit) at step 250,000



LlamaModel is using LlamaSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.
Sampling:  21%|██▏       | 213/1000 [00:06<00:22, 34.25it/s]


In [9]:
#!/usr/bin/env python3
"""
Download Chatterbox model files from Hugging Face Hub
"""

import os
from pathlib import Path
from huggingface_hub import hf_hub_download
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def download_chatterbox_models(
    models_dir: str = "models",
    repo_id: str = "ResembleAI/chatterbox",
    cache_dir: str = None
):
    """
    Download all Chatterbox model files from Hugging Face Hub
    
    Args:
        models_dir: Local directory to save the model files
        repo_id: Hugging Face repository ID
        cache_dir: Optional cache directory for Hugging Face downloads
    """
    
    # Create models directory if it doesn't exist
    models_path = Path(models_dir)
    models_path.mkdir(parents=True, exist_ok=True)
    
    # List of required model files
    required_files = [
        "ve.safetensors",           # Voice Encoder weights
        "t3_cfg.safetensors",       # T3 transformer model weights
        "s3gen.safetensors",        # Speech generator weights  
        "tokenizer.json",           # Text tokenizer configuration
    ]
    
    # Optional files
    optional_files = [
        "conds.pt",                 # Conditioning data
    ]
    
    logger.info(f"Downloading Chatterbox model files to: {models_path.absolute()}")
    logger.info(f"Repository: {repo_id}")
    
    # Download required files
    for filename in required_files:
        try:
            logger.info(f"Downloading {filename}...")
            local_path = hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                local_dir=models_path,
                local_dir_use_symlinks=False,
                cache_dir=cache_dir
            )
            logger.info(f"✓ Downloaded {filename} to {local_path}")
            
        except Exception as e:
            logger.error(f"✗ Failed to download {filename}: {e}")
            raise
    
    # Download optional files (don't fail if missing)
    for filename in optional_files:
        try:
            logger.info(f"Downloading optional file {filename}...")
            local_path = hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                local_dir=models_path,
                local_dir_use_symlinks=False,
                cache_dir=cache_dir
            )
            logger.info(f"✓ Downloaded {filename} to {local_path}")
            
        except Exception as e:
            logger.warning(f"! Optional file {filename} not found or failed to download: {e}")
    
    # Verify all required files exist
    missing_files = []
    for filename in required_files:
        file_path = models_path / filename
        if not file_path.exists():
            missing_files.append(filename)
    
    if missing_files:
        raise FileNotFoundError(f"Missing required files: {missing_files}")
    
    logger.info("✓ All model files downloaded successfully!")
    logger.info(f"Model directory: {models_path.absolute()}")
    
    # Print file sizes for verification
    logger.info("\nDownloaded files:")
    for file_path in models_path.glob("*"):
        if file_path.is_file():
            size_mb = file_path.stat().st_size / (1024 * 1024)
            logger.info(f"  {file_path.name}: {size_mb:.2f} MB")
    
    return models_path

def main():
    """Main function with command line argument support"""

    
    try:
        models_path = download_chatterbox_models(
            models_dir="/home/user/voice/chatterbox/models",
            repo_id="ResembleAI/chatterbox",
        )
        print(f"\n🎉 Success! Model files downloaded to: {models_path.absolute()}")
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        exit(1)

if __name__ == "__main__":
    main()

/home/user/anaconda3/envs/chatterbox/lib/python3.11/site-packages/huggingface_hub/file_download.py:980: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(



🎉 Success! Model files downloaded to: /home/user/voice/chatterbox/models


In [2]:
from safetensors.torch import load_file, save_file
import torch
import numpy as np

# Load safetensors
model_path = "/home/user/voice/chatterbox/models/t3_cfg.safetensors"
model_weights = load_file(model_path)

# # Define new shape
# old_vocab_size, hidden_dim = 704, 1024
# new_vocab_size = 2000

# for key in ['text_head.weight', 'text_emb.weight']:
#     print(f"Processing {key}...")

#     old_tensor = weights[key]  # Shape: [704, 1024]
#     assert old_tensor.shape[0] == old_vocab_size and old_tensor.shape[1] == hidden_dim

#     # Create new tensor and copy old values
#     new_tensor = torch.zeros((new_vocab_size, hidden_dim), dtype=old_tensor.dtype)
#     new_tensor[:old_vocab_size] = old_tensor

#     # Initialize new rows (common: normal with same std as pretrained, or xavier)
#     std = old_tensor.std().item()
#     mean = old_tensor.mean().item()
#     torch.nn.init.normal_(new_tensor[old_vocab_size:], mean=mean, std=std)

#     # Replace in weights dict
#     weights[key] = new_tensor

# Save to new safetensors file
# save_file(weights, "t3_cfg_extended.safetensors")
# print("Embedding extension complete. Saved as t3_cfg_extended.safetensors.")


In [15]:
print(model_weights["text_head.weight"].shape)

torch.Size([704, 1024])


In [16]:
print(model_weights["text_emb.weight"].shape)

torch.Size([704, 1024])


In [ ]:

model_weights.keys()

dict_keys(['cond_enc.emotion_adv_fc.weight', 'cond_enc.perceiver.attn.norm.bias', 'cond_enc.perceiver.attn.norm.weight', 'cond_enc.perceiver.attn.proj_out.bias', 'cond_enc.perceiver.attn.proj_out.weight', 'cond_enc.perceiver.attn.to_k.bias', 'cond_enc.perceiver.attn.to_k.weight', 'cond_enc.perceiver.attn.to_q.bias', 'cond_enc.perceiver.attn.to_q.weight', 'cond_enc.perceiver.attn.to_v.bias', 'cond_enc.perceiver.attn.to_v.weight', 'cond_enc.perceiver.pre_attention_query', 'cond_enc.spkr_enc.bias', 'cond_enc.spkr_enc.weight', 'speech_emb.weight', 'speech_head.weight', 'speech_pos_emb.emb.weight', 'text_emb.weight', 'text_head.weight', 'text_pos_emb.emb.weight', 'tfmr.embed_tokens.weight', 'tfmr.layers.0.input_layernorm.weight', 'tfmr.layers.0.mlp.down_proj.weight', 'tfmr.layers.0.mlp.gate_proj.weight', 'tfmr.layers.0.mlp.up_proj.weight', 'tfmr.layers.0.post_attention_layernorm.weight', 'tfmr.layers.0.self_attn.k_proj.weight', 'tfmr.layers.0.self_attn.o_proj.weight', 'tfmr.layers.0.self_at

In [11]:
model.t3.tfmr

LlamaModel(
  (embed_tokens): Embedding(8, 1024)
  (layers): ModuleList(
    (0-29): 30 x LlamaDecoderLayer(
      (self_attn): LlamaSdpaAttention(
        (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (rotary_emb): LlamaRotaryEmbedding()
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=1024, out_features=4096, bias=False)
        (up_proj): Linear(in_features=1024, out_features=4096, bias=False)
        (down_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm((1024,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((1024,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((1024,), eps=1e-05)
  (rotary_emb): LlamaRotaryEmbedding()
)

In [15]:
model.ve

VoiceEncoder(
  (lstm): LSTM(40, 256, num_layers=3, batch_first=True)
  (proj): Linear(in_features=256, out_features=256, bias=True)
)